In [36]:
# Import libraries
import pandas as pd
import numpy as np
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Grid
from pyecharts.commons.utils import JsCode

In [37]:
#%% Load and prepare data

# Load on-chain transaction data
bitcoin_tx = pd.read_pickle('files/bitquery/bitcoin_tx.pkl')
ethereum_tx = pd.read_pickle('files/bitquery/ethereum_tx.pkl')
bnb_smart_tx = pd.read_pickle('files/bitquery/bnb_smart_tx.pkl')
avalanche_tx = pd.read_pickle('files/bitquery/avalanche_tx.pkl')
ripple_tx = pd.read_pickle('files/bitquery/ripple_tx.pkl')

onchain_df = pd.concat([bitcoin_tx, ethereum_tx, bnb_smart_tx, avalanche_tx, ripple_tx], ignore_index=True)

chain_to_symbol = {
    'bitcoin': 'BTC', 'ethereum': 'ETH', 'bnb_smart': 'BNB',
    'avalanche': 'AVAX', 'ripple': 'XRP'
}
onchain_df['symbols'] = onchain_df['chain'].map(chain_to_symbol)
onchain_df['date'] = pd.to_datetime(onchain_df['date'])

asset_network_pairs = {
    'BTC': 'bitcoin', 'ETH': 'ethereum', 'BNB': 'bnb_smart',
    'AVAX': 'avalanche', 'XRP': 'ripple'
}

# Aggregate to monthly
onchain_df['timestamp'] = onchain_df['date'].dt.to_period('M')
monthly_tx = onchain_df.groupby(['symbols', 'timestamp'])['tx_count'].sum().reset_index()
monthly_tx.rename(columns={'tx_count': 'monthly_tx_count'}, inplace=True)

# Load price data
avg_df = pd.read_pickle('files/avg_df.pkl')
avg_df['timestamp'] = avg_df['timestamp'].dt.to_timestamp().dt.to_period('M')
avg_df = avg_df.groupby(['symbols', 'timestamp']).agg({'average_price': 'mean'}).reset_index()

monthly_tx['timestamp'] = monthly_tx['timestamp'].astype(str)
avg_df['timestamp'] = avg_df['timestamp'].astype(str)

merged_df = pd.merge(avg_df, monthly_tx, on=['symbols', 'timestamp'], how='inner')
merged_df = merged_df.sort_values(['symbols', 'timestamp']).reset_index(drop=True)

# Compute correlations
corr_results = {}
for symbol, chain in asset_network_pairs.items():
    sd = merged_df[merged_df['symbols'] == symbol].copy()
    if len(sd) < 3:
        continue
    pearson = sd['average_price'].corr(sd['monthly_tx_count'])
    rank_price = sd['average_price'].rank()
    rank_tx = sd['monthly_tx_count'].rank()
    spearman = rank_price.corr(rank_tx)
    corr_results[symbol] = {'pearson': round(pearson, 4), 'spearman': round(spearman, 4)}

print('Data ready.')
print(f'Total merged records: {len(merged_df)}')

Data ready.
Total merged records: 411


C:\Users\Matheus\AppData\Local\Temp\ipykernel_32616\2586771944.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  avg_df = avg_df.groupby(['symbols', 'timestamp']).agg({'average_price': 'mean'}).reset_index()


In [38]:
#%% JS abbreviation formatter

abbr_formatter = JsCode("""
function (value) {
    if (value >= 1e12) { return +(value/1e12).toFixed(1) + 'T'; }
    if (value >= 1e9)  { return +(value/1e9).toFixed(1)  + 'B'; }
    if (value >= 1e6)  { return +(value/1e6).toFixed(1)  + 'M'; }
    if (value >= 1e3)  { return +(value/1e3).toFixed(1)  + 'K'; }
    return value;
}
""")

price_formatter = JsCode("""
function (value) {
    if (value >= 1e6)  { return '$' + +(value/1e6).toFixed(1)  + 'M'; }
    if (value >= 1e3)  { return '$' + +(value/1e3).toFixed(1)  + 'K'; }
    return '$' + value;
}
""")

In [39]:
#%% Color palette

color_map = {
    'BTC': ['#FC922F', '#FFC88A'],   # Orange
    'ETH': ['#626AFF', '#A8ADFF'],   # Blue
    'BNB': ['#FFCF3D', '#FFE78A'],   # Yellow
    'AVAX': ['#FF3A3A', '#FF8A8A'],  # Red
    'XRP': ['#DCDCDC', '#A0A0A0'],   # Grey
}

In [40]:
#%% BTC Price vs Bitcoin Network Activity

symbol, chain = 'BTC', 'bitcoin'
sd = merged_df[merged_df['symbols'] == symbol].sort_values('timestamp')
x_axis = sd['timestamp'].tolist()
prices = sd['average_price'].round(2).tolist()
tx_counts = sd['monthly_tx_count'].astype(int).tolist()
colors = color_map[symbol]
p_val = corr_results[symbol]['pearson']
s_val = corr_results[symbol]['spearman']

bar = Bar()
bar.add_xaxis(x_axis)
bar.add_yaxis(
    f'{chain.upper()} Transactions', tx_counts,
    label_opts=opts.LabelOpts(is_show=False),
    itemstyle_opts=opts.ItemStyleOpts(color=colors[1], opacity=0.5),
    yaxis_index=0,
)
bar.extend_axis(
    yaxis=opts.AxisOpts(
        name=f'{symbol} Price (USD)',
        position='right',
        axislabel_opts=opts.LabelOpts(formatter=price_formatter),
        splitline_opts=opts.SplitLineOpts(is_show=False),
    )
)

line = Line()
line.add_xaxis(x_axis)
line.add_yaxis(
    f'{symbol} Price', prices,
    label_opts=opts.LabelOpts(is_show=False),
    itemstyle_opts=opts.ItemStyleOpts(color=colors[0]),
    linestyle_opts=opts.LineStyleOpts(width=2),
    yaxis_index=1,
)

bar.overlap(line)
bar.set_global_opts(
    title_opts=opts.TitleOpts(
        title=f'{symbol} Price vs {chain.upper()} Network Activity',
        subtitle=f'Pearson: {p_val}  |  Spearman: {s_val}',
    ),
    tooltip_opts=opts.TooltipOpts(trigger='axis', axis_pointer_type='cross'),
    datazoom_opts=opts.DataZoomOpts(type_='slider', range_start=0, range_end=100),
    yaxis_opts=opts.AxisOpts(
        name='Tx Count',
        axislabel_opts=opts.LabelOpts(formatter=abbr_formatter),
    ),
    xaxis_opts=opts.AxisOpts(splitline_opts=opts.SplitLineOpts(is_show=False)),
    legend_opts=opts.LegendOpts(pos_left='right', is_show=False),
)

grid = Grid(init_opts=opts.InitOpts(theme='dark', bg_color='rgba(0,0,0,0)', width='100%', height='535px'))
grid.add(bar, grid_opts=opts.GridOpts(pos_top='80px'), is_control_axis_index=True)
grid.render('echarts/correlation_btc.html', template_name='simple_chart.html')
grid.render_notebook()

In [41]:
#%% ETH Price vs Ethereum Network Activity

symbol, chain = 'ETH', 'ethereum'
sd = merged_df[merged_df['symbols'] == symbol].sort_values('timestamp')
x_axis = sd['timestamp'].tolist()
prices = sd['average_price'].round(2).tolist()
tx_counts = sd['monthly_tx_count'].astype(int).tolist()
colors = color_map[symbol]
p_val = corr_results[symbol]['pearson']
s_val = corr_results[symbol]['spearman']

bar = Bar()
bar.add_xaxis(x_axis)
bar.add_yaxis(
    f'{chain.upper()} Transactions', tx_counts,
    label_opts=opts.LabelOpts(is_show=False),
    itemstyle_opts=opts.ItemStyleOpts(color=colors[1], opacity=0.5),
    yaxis_index=0,
)
bar.extend_axis(
    yaxis=opts.AxisOpts(
        name=f'{symbol} Price (USD)',
        position='right',
        axislabel_opts=opts.LabelOpts(formatter=price_formatter),
        splitline_opts=opts.SplitLineOpts(is_show=False),
    )
)

line = Line()
line.add_xaxis(x_axis)
line.add_yaxis(
    f'{symbol} Price', prices,
    label_opts=opts.LabelOpts(is_show=False),
    itemstyle_opts=opts.ItemStyleOpts(color=colors[0]),
    linestyle_opts=opts.LineStyleOpts(width=2),
    yaxis_index=1,
)

bar.overlap(line)
bar.set_global_opts(
    title_opts=opts.TitleOpts(
        title=f'{symbol} Price vs {chain.upper()} Network Activity',
        subtitle=f'Pearson: {p_val}  |  Spearman: {s_val}',
    ),
    tooltip_opts=opts.TooltipOpts(trigger='axis', axis_pointer_type='cross'),
    datazoom_opts=opts.DataZoomOpts(type_='slider', range_start=0, range_end=100),
    yaxis_opts=opts.AxisOpts(
        name='Tx Count',
        axislabel_opts=opts.LabelOpts(formatter=abbr_formatter),
    ),
    xaxis_opts=opts.AxisOpts(splitline_opts=opts.SplitLineOpts(is_show=False)),
    legend_opts=opts.LegendOpts(pos_left='right', is_show=False),
)

grid = Grid(init_opts=opts.InitOpts(theme='dark', bg_color='rgba(0,0,0,0)', width='100%', height='535px'))
grid.add(bar, grid_opts=opts.GridOpts(pos_top='80px'), is_control_axis_index=True)
grid.render('echarts/correlation_eth.html', template_name='simple_chart.html')
grid.render_notebook()

In [42]:
#%% BNB Price vs BNB_SMART Network Activity

symbol, chain = 'BNB', 'bnb_smart'
sd = merged_df[merged_df['symbols'] == symbol].sort_values('timestamp')
x_axis = sd['timestamp'].tolist()
prices = sd['average_price'].round(2).tolist()
tx_counts = sd['monthly_tx_count'].astype(int).tolist()
colors = color_map[symbol]
p_val = corr_results[symbol]['pearson']
s_val = corr_results[symbol]['spearman']

bar = Bar()
bar.add_xaxis(x_axis)
bar.add_yaxis(
    f'{chain.upper()} Transactions', tx_counts,
    label_opts=opts.LabelOpts(is_show=False),
    itemstyle_opts=opts.ItemStyleOpts(color=colors[1], opacity=0.5),
    yaxis_index=0,
)
bar.extend_axis(
    yaxis=opts.AxisOpts(
        name=f'{symbol} Price (USD)',
        position='right',
        axislabel_opts=opts.LabelOpts(formatter=price_formatter),
        splitline_opts=opts.SplitLineOpts(is_show=False),
    )
)

line = Line()
line.add_xaxis(x_axis)
line.add_yaxis(
    f'{symbol} Price', prices,
    label_opts=opts.LabelOpts(is_show=False),
    itemstyle_opts=opts.ItemStyleOpts(color=colors[0]),
    linestyle_opts=opts.LineStyleOpts(width=2),
    yaxis_index=1,
)

bar.overlap(line)
bar.set_global_opts(
    title_opts=opts.TitleOpts(
        title=f'{symbol} Price vs {chain.upper()} Network Activity',
        subtitle=f'Pearson: {p_val}  |  Spearman: {s_val}',
    ),
    tooltip_opts=opts.TooltipOpts(trigger='axis', axis_pointer_type='cross'),
    datazoom_opts=opts.DataZoomOpts(type_='slider', range_start=0, range_end=100),
    yaxis_opts=opts.AxisOpts(
        name='Tx Count',
        axislabel_opts=opts.LabelOpts(formatter=abbr_formatter),
    ),
    xaxis_opts=opts.AxisOpts(splitline_opts=opts.SplitLineOpts(is_show=False)),
    legend_opts=opts.LegendOpts(pos_left='right', is_show=False),
)

grid = Grid(init_opts=opts.InitOpts(theme='dark', bg_color='rgba(0,0,0,0)', width='100%', height='535px'))
grid.add(bar, grid_opts=opts.GridOpts(pos_top='80px'), is_control_axis_index=True)
grid.render('echarts/correlation_bnb.html', template_name='simple_chart.html')
grid.render_notebook()

In [43]:
#%% AVAX Price vs Avalanche Network Activity

symbol, chain = 'AVAX', 'avalanche'
sd = merged_df[merged_df['symbols'] == symbol].sort_values('timestamp')
x_axis = sd['timestamp'].tolist()
prices = sd['average_price'].round(2).tolist()
tx_counts = sd['monthly_tx_count'].astype(int).tolist()
colors = color_map[symbol]
p_val = corr_results[symbol]['pearson']
s_val = corr_results[symbol]['spearman']

bar = Bar()
bar.add_xaxis(x_axis)
bar.add_yaxis(
    f'{chain.upper()} Transactions', tx_counts,
    label_opts=opts.LabelOpts(is_show=False),
    itemstyle_opts=opts.ItemStyleOpts(color=colors[1], opacity=0.5),
    yaxis_index=0,
)
bar.extend_axis(
    yaxis=opts.AxisOpts(
        name=f'{symbol} Price (USD)',
        position='right',
        axislabel_opts=opts.LabelOpts(formatter=price_formatter),
        splitline_opts=opts.SplitLineOpts(is_show=False),
    )
)

line = Line()
line.add_xaxis(x_axis)
line.add_yaxis(
    f'{symbol} Price', prices,
    label_opts=opts.LabelOpts(is_show=False),
    itemstyle_opts=opts.ItemStyleOpts(color=colors[0]),
    linestyle_opts=opts.LineStyleOpts(width=2),
    yaxis_index=1,
)

bar.overlap(line)
bar.set_global_opts(
    title_opts=opts.TitleOpts(
        title=f'{symbol} Price vs {chain.upper()} Network Activity',
        subtitle=f'Pearson: {p_val}  |  Spearman: {s_val}',
    ),
    tooltip_opts=opts.TooltipOpts(trigger='axis', axis_pointer_type='cross'),
    datazoom_opts=opts.DataZoomOpts(type_='slider', range_start=0, range_end=100),
    yaxis_opts=opts.AxisOpts(
        name='Tx Count',
        axislabel_opts=opts.LabelOpts(formatter=abbr_formatter),
    ),
    xaxis_opts=opts.AxisOpts(splitline_opts=opts.SplitLineOpts(is_show=False)),
    legend_opts=opts.LegendOpts(pos_left='right', is_show=False),
)

grid = Grid(init_opts=opts.InitOpts(theme='dark', bg_color='rgba(0,0,0,0)', width='100%', height='535px'))
grid.add(bar, grid_opts=opts.GridOpts(pos_top='80px'), is_control_axis_index=True)
grid.render('echarts/correlation_avax.html', template_name='simple_chart.html')
grid.render_notebook()

In [44]:
#%% XRP Price vs Ripple Network Activity

symbol, chain = 'XRP', 'ripple'
sd = merged_df[merged_df['symbols'] == symbol].sort_values('timestamp')
x_axis = sd['timestamp'].tolist()
prices = sd['average_price'].round(2).tolist()
tx_counts = sd['monthly_tx_count'].astype(int).tolist()
colors = color_map[symbol]
p_val = corr_results[symbol]['pearson']
s_val = corr_results[symbol]['spearman']

bar = Bar()
bar.add_xaxis(x_axis)
bar.add_yaxis(
    f'{chain.upper()} Transactions', tx_counts,
    label_opts=opts.LabelOpts(is_show=False),
    itemstyle_opts=opts.ItemStyleOpts(color=colors[1], opacity=0.5),
    yaxis_index=0,
)
bar.extend_axis(
    yaxis=opts.AxisOpts(
        name=f'{symbol} Price (USD)',
        position='right',
        axislabel_opts=opts.LabelOpts(formatter=price_formatter),
        splitline_opts=opts.SplitLineOpts(is_show=False),
    )
)

line = Line()
line.add_xaxis(x_axis)
line.add_yaxis(
    f'{symbol} Price', prices,
    label_opts=opts.LabelOpts(is_show=False),
    itemstyle_opts=opts.ItemStyleOpts(color=colors[0]),
    linestyle_opts=opts.LineStyleOpts(width=2),
    yaxis_index=1,
)

bar.overlap(line)
bar.set_global_opts(
    title_opts=opts.TitleOpts(
        title=f'{symbol} Price vs {chain.upper()} Network Activity',
        subtitle=f'Pearson: {p_val}  |  Spearman: {s_val}',
    ),
    tooltip_opts=opts.TooltipOpts(trigger='axis', axis_pointer_type='cross'),
    datazoom_opts=opts.DataZoomOpts(type_='slider', range_start=0, range_end=100),
    yaxis_opts=opts.AxisOpts(
        name='Tx Count',
        axislabel_opts=opts.LabelOpts(formatter=abbr_formatter),
    ),
    xaxis_opts=opts.AxisOpts(splitline_opts=opts.SplitLineOpts(is_show=False)),
    legend_opts=opts.LegendOpts(pos_left='right', is_show=False),
)

grid = Grid(init_opts=opts.InitOpts(theme='dark', bg_color='rgba(0,0,0,0)', width='100%', height='535px'))
grid.add(bar, grid_opts=opts.GridOpts(pos_top='80px'), is_control_axis_index=True)
grid.render('echarts/correlation_xrp.html', template_name='simple_chart.html')
grid.render_notebook()